In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt

In [ ]:
report = pd.read_csv('data/clinical_report_clean.csv')
ml_ready = pd.read_csv('data/ml_dataset_ready.csv')
cols_to_use = ml_ready.columns.difference(report.columns).tolist() + ['patient_id']
full_data = pd.merge(report, ml_ready[cols_to_use], on='patient_id', how='inner')

excluded_cols = [
    'patient_id', 'target_high_risk', 'target_risk_category', 
    'predicted_cluster', 'cluster_1_prob', 'cluster_2_prob', 'cluster_3_prob',
    'группа наблюдения', 'время измерения', 
    'kzs_count' 
]

X_train = full_data.drop(columns=[c for c in excluded_cols if c in full_data.columns])
y_train = full_data['target_high_risk']

# Hypothesis H13: Clinical Interpretability (SHAP Analysis)
**Hypothesis Formulation:** The high prediction accuracy of the model is not a consequence of overfitting or noise, but is based on clinically significant biomarkers consistent with established medical protocols.

### Analysis Objectives:
1. **Global Interpretability:** Identification of key physiological risk factors using SHAP (SHapley Additive exPlanations) values.
2. **Feature Interaction:** Analysis of non-linear interactions between biomarkers (e.g., relationship between hypertension duration and median diastolic pressure) and their impact on risk assessment.
3. **Clinical Validation:** Ensuring transparency of the model's decision-making process to provide interpretable data to medical professionals.

### Visualization:
* **Summary Plot:** Distribution of each feature's impact on model output.
* **Dependence Plot:** Detailed analysis of the relationship between the main predictor and the estimated risk probability.

In [ ]:
explainer = shap.TreeExplainer(model_final)

X_sample = X_test_final.sample(min(500, len(X_test_final)), random_state=42)
shap_values = explainer.shap_values(X_sample)

importances = pd.Series(model_final.feature_importances_, index=X_test_final.columns)
print("Key Risk Predictors (Top 5):")
print(importances.sort_values(ascending=False).head(5))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP Feature Importance (Global Impact)")
plt.show()

best_feature = importances.idxmax()

shap.dependence_plot(best_feature, shap_values, X_sample, interaction_index="auto", show=False)
plt.title(f"SHAP Interaction: {best_feature}")
plt.show()